In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [4]:
spitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap = 100)
splitted_data = spitter.split_documents(docs)

In [5]:
len(splitted_data)

28

In [7]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
)

vectorstore = InMemoryVectorStore.from_documents(
    documents = splitted_data, 
    embedding = embeddings
)

In [10]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """

    print("Tool Called: ", query)
    docs = vectorstore.similarity_search(query=query, k = 4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"
    
    return context

In [11]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [12]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [13]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [17]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  patient name doctor name medical report


In [15]:
result = response["messages"][-1].content

In [16]:
print(result)

**Patient:** Ms. Nikita Chudhary  
**Doctor:** Dr. Nitin Nahar
